In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from toolbox.production_range import build_xt_table
from toolbox.metrics.kerma_coeffs import k_coeff_pGy_cm2_from_GeV
from model import Model

# prefac2: (1 - exp(-(EP/E_th)^n))  — threshold suppression only, no power law
param_names_m4 = [
    'A1', 'gamma1', 'Sigma_t', 'Sigma', 'd1', 'a', 'w_c',
    'A2', 'gamma2', 'd2', 'kappa2', 'Epk',
    'A3', 'gamma3', 'kappa3',
    'A4', 'gamma4', 'kappa4',
    'E_th', 'n',
]

inf = np.inf
lower_bounds_m4 = [0,    0,    0,    0,    -inf, 0,   0,
                   0,    0,    -inf, 0,    0,
                   0,    0,    0,
                   0,    0,    0,
                   0,    0]
upper_bounds_m4 = [inf,  9,    inf,  5,    inf,  inf, inf,
                   inf,  9,    10,   6,    inf,
                   inf,  9,    6,
                   inf,  9,    6,
                   inf,  inf]

# p0: previous converged prefac2 fit
_df_prev = pd.read_csv('fitting_params/proton_prefac2.csv', index_col=0)
_p0      = [float(_df_prev.loc['opt params', n]) for n in param_names_m4]
print('p0 loaded from fitting_params/proton_prefac2.csv')

EPK_FIXED = 0.004
_epk_idx   = param_names_m4.index('Epk')
_lo_no_epk = [lower_bounds_m4[i] for i, n in enumerate(param_names_m4) if n != 'Epk']
_hi_no_epk = [upper_bounds_m4[i] for i, n in enumerate(param_names_m4) if n != 'Epk']
_p0_no_epk = [_p0[i] for i, n in enumerate(param_names_m4) if n != 'Epk']

s           = 'proton'
Ep_all      = np.load(f'npy_data/{s}_energies.npy')
mc_all      = np.load(f'npy_data/{s}_mc.npy')[:, :, :35, ::2]
mc_full     = np.load(f'npy_data/{s}_mc.npy')[:, :, :35, :]
z           = np.load('npy_data/z.npy')
rho_35      = np.load('npy_data/rho.npy')[:35]
en_low      = np.load('npy_data/en_low.npy')[::2]
en_low_full = np.load('npy_data/en_low.npy')
en_upp_full = np.load('npy_data/en_upp.npy')

ep_indices = np.linspace(0, len(Ep_all)-1, 8, dtype=int)
Ep_fit     = Ep_all[ep_indices]
mc_fit     = mc_all[ep_indices]
print(f'Fitting Ep: {np.round(Ep_fit, 1)}')

nEp, nz, nr, nE = mc_fit.shape
XT   = build_xt_table(s, Ep_fit, en_low)
iEp  = np.repeat(np.arange(nEp), nz * nr * nE)
iEn  = np.tile(np.arange(nE), nEp * nz * nr)
M    = Model(species=s, XT=XT, iEp=iEp, iEn=iEn)

EP, Z, R, En = np.meshgrid(Ep_fit, z, rho_35, en_low, indexing='ij')
EP, Z, R, En = EP.flatten(), Z.flatten(), R.flatten(), En.flatten()
MC = mc_fit.flatten()

def model_fixed_epk_m4(X, A1, gamma1, Sigma_t, Sigma, d1, a, w_c,
                        A2, gamma2, d2, kappa2,
                        A3, gamma3, kappa3,
                        A4, gamma4, kappa4,
                        E_th, n):
    return M.spectral_energy_fluence_prefac2(X,
        A1, gamma1, Sigma_t, Sigma, d1, a, w_c,
        A2, gamma2, d2, kappa2, EPK_FIXED,
        A3, gamma3, kappa3,
        A4, gamma4, kappa4,
        E_th, n)

print(f'Fitting {nEp} energies, {nz} z-points ({MC.shape[0]:,} data points), Epk fixed={EPK_FIXED}...')
popt_m4, _ = curve_fit(
    model_fixed_epk_m4,
    (Z, R, En, EP), MC,
    p0=_p0_no_epk,
    bounds=(_lo_no_epk, _hi_no_epk),
    method='trf',
    max_nfev=10000,
)

_popt_m4_full = list(popt_m4)
_popt_m4_full.insert(_epk_idx, EPK_FIXED)
popts_m4 = pd.DataFrame([_popt_m4_full], columns=param_names_m4, index=['opt params'])
popts_m4.to_csv('fitting_params/proton_prefac2.csv')
print(popts_m4.T)

# --- metrics over fitted energies ---
dE      = en_upp_full - en_low_full
k_c     = k_coeff_pGy_cm2_from_GeV(en_low_full)
nE_full = len(en_low_full)
wfrd_list, wkrd_list = [], []

for i, ep_val in enumerate(Ep_fit):
    Ep_i  = Ep_fit[i:i+1]
    mc_i  = mc_full[ep_indices[i]:ep_indices[i]+1]
    nz_f, nr_f = len(z), len(rho_35)
    XT_i  = build_xt_table(s, Ep_i, en_low_full)
    iEp_i = np.repeat(np.arange(1), nz_f * nr_f * nE_full)
    iEn_i = np.tile(np.arange(nE_full), nz_f * nr_f)
    M_i   = Model(species=s, XT=XT_i, iEp=iEp_i, iEn=iEn_i)
    EP_i, Z_i, R_i, En_i = np.meshgrid(Ep_i, z, rho_35, en_low_full, indexing='ij')
    EP_i, Z_i, R_i, En_i = EP_i.flatten(), Z_i.flatten(), R_i.flatten(), En_i.flatten()
    pred    = M_i.spectral_energy_fluence_prefac2((Z_i, R_i, En_i, EP_i), *_popt_m4_full).reshape(1, nz_f, nr_f, nE_full)
    phi_mc  = np.sum(mc_i * dE, axis=3)
    phi_mod = np.sum(pred * dE, axis=3)
    k_mc    = np.sum(mc_i * dE * k_c, axis=3)
    k_mod   = np.sum(pred * dE * k_c, axis=3)
    wf = (np.sum((phi_mod-phi_mc)*rho_35,axis=(1,2)) / np.sum(phi_mc*rho_35,axis=(1,2)))[0]
    wk = (np.sum((k_mod  -k_mc  )*rho_35,axis=(1,2)) / np.sum(k_mc *rho_35,axis=(1,2)))[0]
    wfrd_list.append(wf)
    wkrd_list.append(wk)
    print(f'  Ep={ep_val:.1f} MeV:  D_Phi={wf*100:.1f}%   D_K={wk*100:.1f}%')

print(f'Fluence  mean|D|: {np.mean(np.abs(wfrd_list))*100:.1f}%   median|D|: {np.median(np.abs(wfrd_list))*100:.1f}%')
print(f'Kerma    mean|D|: {np.mean(np.abs(wkrd_list))*100:.1f}%   median|D|: {np.median(np.abs(wkrd_list))*100:.1f}%')


In [15]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from toolbox.production_range import build_xt_table
from toolbox.metrics.kerma_coeffs import k_coeff_pGy_cm2_from_GeV
from model import Model

param_names_m4 = [
    'A1', 'gamma1', 'Sigma_t', 'Sigma', 'd1', 'a', 'w_c',
    'A2', 'gamma2', 'd2', 'kappa2', 'Epk',
    'A3', 'gamma3', 'kappa3',
    'A4', 'gamma4', 'kappa4',
    'E_th', 'n',
]

inf = np.inf
lower_bounds_m4 = [0,    0,    0,    0,    -inf, 0,   0,
                   0,    0,    -inf, 0,    0,
                   0,    0,    0,
                   0,    0,    0,
                   0,    0]
upper_bounds_m4 = [inf,  9,    inf,  5,    inf,  inf, inf,
                   inf,  9,    10,   6,    inf,
                   inf,  9,    6,
                   inf,  9,    6,
                   inf,  inf]

# p0: previous converged prefac2 fit
_df_prev = pd.read_csv('fitting_params/carbon_prefac2.csv', index_col=0)
_p0      = [float(_df_prev.loc['opt params', n]) for n in param_names_m4]
print('p0 loaded from fitting_params/carbon_prefac2.csv')

EPK_FIXED  = 0.004
_epk_idx   = param_names_m4.index('Epk')
_lo_no_epk = [lower_bounds_m4[i] for i, n in enumerate(param_names_m4) if n != 'Epk']
_hi_no_epk = [upper_bounds_m4[i] for i, n in enumerate(param_names_m4) if n != 'Epk']
_p0_no_epk = [_p0[i] for i, n in enumerate(param_names_m4) if n != 'Epk']

s           = 'carbon'
Ep_all      = np.load(f'npy_data/{s}_energies.npy')
mc_all      = np.load(f'npy_data/{s}_mc.npy')[:, :, :35, ::2]
mc_full     = np.load(f'npy_data/{s}_mc.npy')[:, :, :35, :]
z           = np.load('npy_data/z.npy')
rho_35      = np.load('npy_data/rho.npy')[:35]
en_low      = np.load('npy_data/en_low.npy')[::2]
en_low_full = np.load('npy_data/en_low.npy')
en_upp_full = np.load('npy_data/en_upp.npy')

ep_indices = np.linspace(0, len(Ep_all)-1, 8, dtype=int)
Ep_fit     = Ep_all[ep_indices]
mc_fit     = mc_all[ep_indices]
print(f'Fitting Ep: {np.round(Ep_fit, 1)}')

nEp, nz, nr, nE = mc_fit.shape
XT   = build_xt_table(s, Ep_fit, en_low)
iEp  = np.repeat(np.arange(nEp), nz * nr * nE)
iEn  = np.tile(np.arange(nE), nEp * nz * nr)
M    = Model(species=s, XT=XT, iEp=iEp, iEn=iEn)

EP, Z, R, En = np.meshgrid(Ep_fit, z, rho_35, en_low, indexing='ij')
EP, Z, R, En = EP.flatten(), Z.flatten(), R.flatten(), En.flatten()
MC = mc_fit.flatten()

def model_fixed_epk_m4_c(X, A1, gamma1, Sigma_t, Sigma, d1, a, w_c,
                           A2, gamma2, d2, kappa2,
                           A3, gamma3, kappa3,
                           A4, gamma4, kappa4,
                           E_th, n):
    return M.spectral_energy_fluence_prefac2(X,
        A1, gamma1, Sigma_t, Sigma, d1, a, w_c,
        A2, gamma2, d2, kappa2, EPK_FIXED,
        A3, gamma3, kappa3,
        A4, gamma4, kappa4,
        E_th, n)

print(f'Fitting carbon — {nEp} energies, {nz} z-points ({MC.shape[0]:,} data points), Epk fixed={EPK_FIXED}...')
popt_m4_c, _ = curve_fit(
    model_fixed_epk_m4_c,
    (Z, R, En, EP), MC,
    p0=_p0_no_epk,
    bounds=(_lo_no_epk, _hi_no_epk),
    method='trf',
    max_nfev=10000,
)

_popt_m4_c_full = list(popt_m4_c)
_popt_m4_c_full.insert(_epk_idx, EPK_FIXED)
popts_m4_c = pd.DataFrame([_popt_m4_c_full], columns=param_names_m4, index=['opt params'])
popts_m4_c.to_csv('fitting_params/carbon_prefac2.csv')
print(popts_m4_c.T)

# --- metrics over fitted energies ---
dE      = en_upp_full - en_low_full
k_c     = k_coeff_pGy_cm2_from_GeV(en_low_full)
nE_full = len(en_low_full)
wfrd_list, wkrd_list = [], []

for i, ep_val in enumerate(Ep_fit):
    Ep_i  = Ep_fit[i:i+1]
    mc_i  = mc_full[ep_indices[i]:ep_indices[i]+1]
    nz_f, nr_f = len(z), len(rho_35)
    XT_i  = build_xt_table(s, Ep_i, en_low_full)
    iEp_i = np.repeat(np.arange(1), nz_f * nr_f * nE_full)
    iEn_i = np.tile(np.arange(nE_full), nz_f * nr_f)
    M_i   = Model(species=s, XT=XT_i, iEp=iEp_i, iEn=iEn_i)
    EP_i, Z_i, R_i, En_i = np.meshgrid(Ep_i, z, rho_35, en_low_full, indexing='ij')
    EP_i, Z_i, R_i, En_i = EP_i.flatten(), Z_i.flatten(), R_i.flatten(), En_i.flatten()
    pred    = M_i.spectral_energy_fluence_prefac2((Z_i, R_i, En_i, EP_i), *_popt_m4_c_full).reshape(1, nz_f, nr_f, nE_full)
    phi_mc  = np.sum(mc_i * dE, axis=3)
    phi_mod = np.sum(pred * dE, axis=3)
    k_mc    = np.sum(mc_i * dE * k_c, axis=3)
    k_mod   = np.sum(pred * dE * k_c, axis=3)
    wf = (np.sum((phi_mod-phi_mc)*rho_35,axis=(1,2)) / np.sum(phi_mc*rho_35,axis=(1,2)))[0]
    wk = (np.sum((k_mod  -k_mc  )*rho_35,axis=(1,2)) / np.sum(k_mc *rho_35,axis=(1,2)))[0]
    wfrd_list.append(wf)
    wkrd_list.append(wk)
    print(f'  Ep={ep_val:.1f} MeV/u:  D_Phi={wf*100:.1f}%   D_K={wk*100:.1f}%')

print(f'Fluence  mean|D|: {np.mean(np.abs(wfrd_list))*100:.1f}%   median|D|: {np.median(np.abs(wfrd_list))*100:.1f}%')
print(f'Kerma    mean|D|: {np.mean(np.abs(wkrd_list))*100:.1f}%   median|D|: {np.median(np.abs(wkrd_list))*100:.1f}%')


p0 loaded from carbon_experimental_params_fixed_Epk.csv (xi dropped)
Fitting Ep: [110.6 163.1 212.1 254.7 293.5 330.5 376.7 424.8]
Fitting carbon — 8 energies, 45 z-points (1,575,000 data points), Epk fixed=0.004...
           opt params
A1       2.588663e-01
gamma1   7.783919e-01
Sigma_t  1.251663e-01
Sigma    8.725807e-01
d1      -2.363078e-02
a        9.552676e-01
w_c      1.921002e-01
A2       2.067293e-02
gamma2   1.286843e+00
d2      -6.828624e-01
kappa2   1.884253e-01
Epk      4.000000e-03
A3       1.153618e-12
gamma3   3.443391e-04
kappa3   5.632092e+00
A4       4.623076e+04
gamma4   1.722613e+00
kappa4   5.699015e-03
E_th     2.003110e+03
n        3.793591e-01
  Ep=110.6 MeV/u:  D_Phi=3.3%   D_K=0.9%
  Ep=163.1 MeV/u:  D_Phi=-20.4%   D_K=-21.9%
  Ep=212.1 MeV/u:  D_Phi=-28.9%   D_K=-31.3%
  Ep=254.7 MeV/u:  D_Phi=-33.0%   D_K=-35.7%
  Ep=293.5 MeV/u:  D_Phi=-34.9%   D_K=-37.6%
  Ep=330.5 MeV/u:  D_Phi=-35.9%   D_K=-38.6%
  Ep=376.7 MeV/u:  D_Phi=-34.5%   D_K=-37.3%
  Ep=424.8 